<a href="https://colab.research.google.com/github/hipanky/neha-divide/blob/master/pyspark1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark


In [21]:
from pyspark.sql import SparkSession


In [104]:
from pyspark.sql.functions import col,sum,min,max,count,udf
from pyspark.sql import Row

In [78]:
Spark = SparkSession.builder.getOrCreate()

In [79]:
Spark


In [61]:
data = [
        ['Pankaj Kumar','CA',4000,'Vodafone','M'],
        ['John Meyer','MA',10000,'Airtel','M'],
        ['Mary Sharon','CA',12000,'Vodafone','F'],
        ['John Pandey','MA',8000,'Idea','M'],
        ['Travis Harrington','TX',12000,'Airtel','']
      ]

In [62]:
df = Spark.createDataFrame(data,['Name','State','Salary','Sim','Gender'])

In [63]:
df.show(3)

+------------+-----+------+--------+------+
|        Name|State|Salary|     Sim|Gender|
+------------+-----+------+--------+------+
|Pankaj Kumar|   CA|  4000|Vodafone|     M|
|  John Meyer|   MA| 10000|  Airtel|     M|
| Mary Sharon|   CA| 12000|Vodafone|     F|
+------------+-----+------+--------+------+
only showing top 3 rows


In [28]:
df.select('Name').show(3)

+------------+
|        Name|
+------------+
|Pankaj Kumar|
|  John Meyer|
| Mary Sharon|
+------------+
only showing top 3 rows


In [29]:
df.select('*').show(3)

+------------+-----+------+--------+
|        Name|State|Salary|     Sim|
+------------+-----+------+--------+
|Pankaj Kumar|   CA|  4000|Vodafone|
|  John Meyer|   MA| 10000|  Airtel|
| Mary Sharon|   CA| 12000|Vodafone|
+------------+-----+------+--------+
only showing top 3 rows


In [31]:
df.select(['Name','State']).show(3)

+------------+-----+
|        Name|State|
+------------+-----+
|Pankaj Kumar|   CA|
|  John Meyer|   MA|
| Mary Sharon|   CA|
+------------+-----+
only showing top 3 rows


In [32]:
df.select('Name','State').show(3)

+------------+-----+
|        Name|State|
+------------+-----+
|Pankaj Kumar|   CA|
|  John Meyer|   MA|
| Mary Sharon|   CA|
+------------+-----+
only showing top 3 rows


In [34]:
df.withColumn('Salary_by_50',col('Salary')+50).show(3)

+------------+-----+------+--------+------------+
|        Name|State|Salary|     Sim|Salary_by_50|
+------------+-----+------+--------+------------+
|Pankaj Kumar|   CA|  4000|Vodafone|        4050|
|  John Meyer|   MA| 10000|  Airtel|       10050|
| Mary Sharon|   CA| 12000|Vodafone|       12050|
+------------+-----+------+--------+------------+
only showing top 3 rows


In [37]:
df.withColumn('DoubleSalary',col('Salary')*2).show(3)

+------------+-----+------+--------+------------+
|        Name|State|Salary|     Sim|DoubleSalary|
+------------+-----+------+--------+------------+
|Pankaj Kumar|   CA|  4000|Vodafone|        8000|
|  John Meyer|   MA| 10000|  Airtel|       20000|
| Mary Sharon|   CA| 12000|Vodafone|       24000|
+------------+-----+------+--------+------------+
only showing top 3 rows


In [48]:
df.withColumn(
              "Salary_Cat",
              when(col("Salary") > 10000,"High")
              .otherwise("Low")
              ).show(3)

+------------+-----+------+--------+----------+
|        Name|State|Salary|     Sim|Salary_Cat|
+------------+-----+------+--------+----------+
|Pankaj Kumar|   CA|  4000|Vodafone|       Low|
|  John Meyer|   MA| 10000|  Airtel|       Low|
| Mary Sharon|   CA| 12000|Vodafone|      High|
+------------+-----+------+--------+----------+
only showing top 3 rows


In [52]:
df.withColumn(
              "Salary_Cat",
              when((col("Salary") > 3000) & (col("Salary") < 8000) ,"Low").
              when((col('Salary') > 8000) & (col('Salary') < 10000),'Medium').
              otherwise("High")
              ).show(3)

+------------+-----+------+--------+----------+
|        Name|State|Salary|     Sim|Salary_Cat|
+------------+-----+------+--------+----------+
|Pankaj Kumar|   CA|  4000|Vodafone|       Low|
|  John Meyer|   MA| 10000|  Airtel|      High|
| Mary Sharon|   CA| 12000|Vodafone|      High|
+------------+-----+------+--------+----------+
only showing top 3 rows


In [57]:
df.withColumn("Salary_Cat",
              when((col('Salary')> 3000) & (col('Salary') < 8000),'Low').
              when((col('Salary') >= 8000) & (col('Salary') < 10000),'Medium').
              otherwise('Hign')
              ).show()

+-----------------+-----+------+--------+----------+
|             Name|State|Salary|     Sim|Salary_Cat|
+-----------------+-----+------+--------+----------+
|     Pankaj Kumar|   CA|  4000|Vodafone|       Low|
|       John Meyer|   MA| 10000|  Airtel|      Hign|
|      Mary Sharon|   CA| 12000|Vodafone|      Hign|
|      John Pandey|   MA|  8000|    Idea|    Medium|
|Travis Harrington|   TX| 12000|  Airtel|      Hign|
+-----------------+-----+------+--------+----------+



In [58]:
df.withColumn("Salary_Cat",
              when((df.Salary > 3000) & (df.Salary < 8000),'Low').
              when((df.Salary >= 8000) & (df.Salary < 10000),'Medium').
              otherwise('Hign')
              ).show()

+-----------------+-----+------+--------+----------+
|             Name|State|Salary|     Sim|Salary_Cat|
+-----------------+-----+------+--------+----------+
|     Pankaj Kumar|   CA|  4000|Vodafone|       Low|
|       John Meyer|   MA| 10000|  Airtel|      Hign|
|      Mary Sharon|   CA| 12000|Vodafone|      Hign|
|      John Pandey|   MA|  8000|    Idea|    Medium|
|Travis Harrington|   TX| 12000|  Airtel|      Hign|
+-----------------+-----+------+--------+----------+



In [64]:
df.withColumn('Location',
              when(df.State == 'CA', 'CALIFORNIA').
              when(df.State == 'MA', 'MASS').
              when(df.State == 'TX','TEXAS')
              ).show()



+-----------------+-----+------+--------+------+----------+
|             Name|State|Salary|     Sim|Gender|  Location|
+-----------------+-----+------+--------+------+----------+
|     Pankaj Kumar|   CA|  4000|Vodafone|     M|CALIFORNIA|
|       John Meyer|   MA| 10000|  Airtel|     M|      MASS|
|      Mary Sharon|   CA| 12000|Vodafone|     F|CALIFORNIA|
|      John Pandey|   MA|  8000|    Idea|     M|      MASS|
|Travis Harrington|   TX| 12000|  Airtel|      |     TEXAS|
+-----------------+-----+------+--------+------+----------+



In [71]:
df.withColumn('GenderC',
              when(df.Gender == 'M', 'MALE').
              when(df.Gender == 'F', 'FEMALE').
              when(df.Gender == '','')
              ).show()


+-----------------+-----+------+--------+------+-------+
|             Name|State|Salary|     Sim|Gender|GenderC|
+-----------------+-----+------+--------+------+-------+
|     Pankaj Kumar|   CA|  4000|Vodafone|     M|   MALE|
|       John Meyer|   MA| 10000|  Airtel|     M|   MALE|
|      Mary Sharon|   CA| 12000|Vodafone|     F| FEMALE|
|      John Pandey|   MA|  8000|    Idea|     M|   MALE|
|Travis Harrington|   TX| 12000|  Airtel|      |       |
+-----------------+-----+------+--------+------+-------+



In [69]:
df.select('*',
              when(df.Gender == 'M', 'MALE').
              when(df.State == 'F', 'FEMALE').
              when(df.State == '','')
          ).alias('GenderC').show(3)


+------------+-----+------+--------+------+---------------------------------------------------------------------------------------+
|        Name|State|Salary|     Sim|Gender|CASE WHEN (Gender = M) THEN MALE WHEN (State = F) THEN FEMALE WHEN (State = ) THEN  END|
+------------+-----+------+--------+------+---------------------------------------------------------------------------------------+
|Pankaj Kumar|   CA|  4000|Vodafone|     M|                                                                                   MALE|
|  John Meyer|   MA| 10000|  Airtel|     M|                                                                                   MALE|
| Mary Sharon|   CA| 12000|Vodafone|     F|                                                                                   NULL|
+------------+-----+------+--------+------+---------------------------------------------------------------------------------------+
only showing top 3 rows


In [85]:
bad_movies_list = [Row(None, None, None),
                   Row(None, None, 2020),
                   Row("John Doe", "Awesome Movie", None),
                   Row(None, "Awesome Movie", 2021),
                   Row("Mary Jane", None, 2019),
                   Row("Vikter Duplaix", "Not another teen movie", 2001)]

In [86]:
bad_movies_cols = ['actor_name','movie_title','release_year']

In [87]:
movies = Spark.createDataFrame(bad_movies_list,schema = bad_movies_cols)

In [84]:
movies.show(2)

+----------+-----------+------------+
|actor_name|movie_title|release_year|
+----------+-----------+------------+
|      NULL|       NULL|        NULL|
|      NULL|       NULL|        2020|
+----------+-----------+------------+
only showing top 2 rows


In [90]:
movies.na.drop().show()

+--------------+--------------------+------------+
|    actor_name|         movie_title|release_year|
+--------------+--------------------+------------+
|Vikter Duplaix|Not another teen ...|        2001|
+--------------+--------------------+------------+



In [91]:
movies.na.drop('any').show()

+--------------+--------------------+------------+
|    actor_name|         movie_title|release_year|
+--------------+--------------------+------------+
|Vikter Duplaix|Not another teen ...|        2001|
+--------------+--------------------+------------+



In [92]:
movies.na.drop('all').show()

+--------------+--------------------+------------+
|    actor_name|         movie_title|release_year|
+--------------+--------------------+------------+
|          NULL|                NULL|        2020|
|      John Doe|       Awesome Movie|        NULL|
|          NULL|       Awesome Movie|        2021|
|     Mary Jane|                NULL|        2019|
|Vikter Duplaix|Not another teen ...|        2001|
+--------------+--------------------+------------+



In [94]:
movies.filter(col('actor_name').isNull()!=True).show()

+--------------+--------------------+------------+
|    actor_name|         movie_title|release_year|
+--------------+--------------------+------------+
|      John Doe|       Awesome Movie|        NULL|
|     Mary Jane|                NULL|        2019|
|Vikter Duplaix|Not another teen ...|        2001|
+--------------+--------------------+------------+



In [95]:
movies.filter(col('actor_name').isNull() == True).show()

+----------+-------------+------------+
|actor_name|  movie_title|release_year|
+----------+-------------+------------+
|      NULL|         NULL|        NULL|
|      NULL|         NULL|        2020|
|      NULL|Awesome Movie|        2021|
+----------+-------------+------------+



In [96]:
movies.filter(col('actor_name').isNull() == False).show()

+--------------+--------------------+------------+
|    actor_name|         movie_title|release_year|
+--------------+--------------------+------------+
|      John Doe|       Awesome Movie|        NULL|
|     Mary Jane|                NULL|        2019|
|Vikter Duplaix|Not another teen ...|        2001|
+--------------+--------------------+------------+



In [97]:
movies.filter(col('actor_name').isNull() != False).show()

+----------+-------------+------------+
|actor_name|  movie_title|release_year|
+----------+-------------+------------+
|      NULL|         NULL|        NULL|
|      NULL|         NULL|        2020|
|      NULL|Awesome Movie|        2021|
+----------+-------------+------------+



In [101]:
movies.describe().show(3)

+-------+----------+-----------+-----------------+
|summary|actor_name|movie_title|     release_year|
+-------+----------+-----------+-----------------+
|  count|         3|          3|                4|
|   mean|      NULL|       NULL|          2015.25|
| stddev|      NULL|       NULL|9.535023160258524|
+-------+----------+-----------+-----------------+
only showing top 3 rows


In [103]:
movies.describe('release_year').show()

+-------+-----------------+
|summary|     release_year|
+-------+-----------------+
|  count|                4|
|   mean|          2015.25|
| stddev|9.535023160258524|
|    min|             2001|
|    max|             2021|
+-------+-----------------+



In [136]:
from pyspark.sql.functions import udf

In [127]:
student_data = [('John',98),('Mary',87),('Pankaj',34),('Travis',78),('Andry',67),('Nasu',56),('Kumar',45)]
student_cols = ('name','marks')

In [129]:

student_df = Spark.createDataFrame(student_data,schema=student_cols )

In [113]:
student_df.show(3)

+------+-----+
|  name|marks|
+------+-----+
|  John|   98|
|  Mary|   87|
|Pankaj|   34|
+------+-----+
only showing top 3 rows


In [135]:
@udf
def find_grade(num:int)->str:
  grade = ''
  if num >= 90:
    grade = 'A'
  elif num >= 70:
    grade = 'B'
  elif num >= 50:
    grade = 'C'
  else:
    grade = 'D'
  return grade

In [144]:
student_df.withColumn('grade',find_grade(col('marks'))).show()

+------+-----+-----+
|  name|marks|grade|
+------+-----+-----+
|  John|   98|    A|
|  Mary|   87|    B|
|Pankaj|   34|    D|
|Travis|   78|    B|
| Andry|   67|    C|
|  Nasu|   56|    C|
| Kumar|   45|    D|
+------+-----+-----+



In [157]:
student_df.select(['name','marks']).show(3)

+------+-----+
|  name|marks|
+------+-----+
|  John|   98|
|  Mary|   87|
|Pankaj|   34|
+------+-----+
only showing top 3 rows


In [163]:
student_df.withColumn('marks_by_10',col('marks')+10).show()

+------+-----+-----------+
|  name|marks|marks_by_10|
+------+-----+-----------+
|  John|   98|        108|
|  Mary|   87|         97|
|Pankaj|   34|         44|
|Travis|   78|         88|
| Andry|   67|         77|
|  Nasu|   56|         66|
| Kumar|   45|         55|
+------+-----+-----------+



In [164]:
def get_grade(num:int)->str:
  grade = ''
  if num >= 90:
    grade = 'A'
  elif num >= 70:
    grade = 'B'
  elif num >= 50:
    grade = 'C'
  else:
    grade = 'D'
  return grade

In [165]:
get_grade = udf(get_grade)

In [167]:
student_df.select('*',get_grade(student_df.marks)).show()

+------+-----+----------------+
|  name|marks|get_grade(marks)|
+------+-----+----------------+
|  John|   98|               A|
|  Mary|   87|               B|
|Pankaj|   34|               D|
|Travis|   78|               B|
| Andry|   67|               C|
|  Nasu|   56|               C|
| Kumar|   45|               D|
+------+-----+----------------+



In [170]:
student_df.select('*',get_grade('marks').alias('grade')).show()

+------+-----+-----+
|  name|marks|grade|
+------+-----+-----+
|  John|   98|    A|
|  Mary|   87|    B|
|Pankaj|   34|    D|
|Travis|   78|    B|
| Andry|   67|    C|
|  Nasu|   56|    C|
| Kumar|   45|    D|
+------+-----+-----+



In [176]:
flight = Spark.read.format('csv').option('header','true').option('inderScheme','true').load('flight-summary.csv')

In [174]:
import os

In [175]:
os.listdir()

['.config', 'flight-summary.csv', 'sample_data']

In [177]:
flight.count()

4693

In [178]:
flight.show(10)

+-----------+--------------------+------------+------------+---------+--------------------+----------------+----------+-----+
|origin_code|      origin_airport| origin_city|origin_state|dest_code|        dest_airport|       dest_city|dest_state|count|
+-----------+--------------------+------------+------------+---------+--------------------+----------------+----------+-----+
|        BQN|Rafael Hernández ...|   Aguadilla|          PR|      MCO|Orlando Internati...|         Orlando|        FL|  441|
|        PHL|Philadelphia Inte...|Philadelphia|          PA|      MCO|Orlando Internati...|         Orlando|        FL| 4869|
|        MCI|Kansas City Inter...| Kansas City|          MO|      IAH|George Bush Inter...|         Houston|        TX| 1698|
|        SPI|Abraham Lincoln C...| Springfield|          IL|      ORD|Chicago O'Hare In...|         Chicago|        IL|  998|
|        SNA|John Wayne Airpor...|   Santa Ana|          CA|      PHX|Phoenix Sky Harbo...|         Phoenix|        AZ

In [182]:
flight.printSchema()


root
 |-- origin_code: string (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- dest_code: string (nullable = true)
 |-- dest_airport: string (nullable = true)
 |-- dest_city: string (nullable = true)
 |-- dest_state: string (nullable = true)
 |-- count: string (nullable = true)



In [204]:
from pyspark.sql.functions import count,min,max,sum,mean,countDistinct

In [194]:
movies.count()

6

In [199]:
movies.select('actor_name').count()

6

In [206]:
flight.select(countDistinct('origin_airport'),countDistinct('dest_airport')).show()

+------------------------------+----------------------------+
|count(DISTINCT origin_airport)|count(DISTINCT dest_airport)|
+------------------------------+----------------------------+
|                           322|                         322|
+------------------------------+----------------------------+

